
# URLFusionNet — Final Adaptive-Gate Experiment

This notebook locks the **empirically supported architecture** selected from the diagnostic ablation:

> projected character representation + projected token representation  
> → concatenated gate input  
> → sample-adaptive two-way softmax weighting  
> → weighted character–token fusion  
> → residual refinement  
> → four-class classifier

The explicit product and absolute-difference interaction terms are retained only as a diagnostic ablation because the previous experiment showed that they degraded validation/test performance and caused gate collapse.

## Run protocol

1. Start with `RUN_MODE="GATE_CHECK"`.
2. Confirm that the proposed adaptive gate is finite and genuinely variable.
3. If it passes, change to `RUN_MODE="FINAL"` and rerun all cells.
4. FINAL mode runs seeds **42, 123, 2025**, final ablation, robustness, efficiency, and paper-ready exports.

The previous baseline models are **not retrained**.


In [ ]:

import sys, subprocess
subprocess.check_call([sys.executable,"-m","pip","install","-q","kagglehub"])


## 1. Configuration

In [ ]:

import os, gc, glob, time, random, warnings, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"

RUN_MODE = "FINAL"   # first "GATE_CHECK", then "FINAL"

SPLIT_SEED=42
SEEDS=[42,123,2025]

gpus=tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try: tf.config.experimental.set_memory_growth(gpu,True)
    except Exception: pass

BATCH_SIZE=1024 if gpus else 256
LEARNING_RATE=8e-4

MAX_CHAR_LEN=200
MAX_TOKEN_LEN=40
MAX_CHAR_VOCAB=256
MAX_TOKEN_VOCAB=30000
CHAR_EMB_DIM=32
TOKEN_EMB_DIM=48
CONV_FILTERS=48
LATENT_DIM=96
ATTN_HEADS=2
DROPOUT=0.20
NUM_CLASSES=4

CLASS_NAMES=["benign","defacement","phishing","malware"]
LABEL_TO_ID={n:i for i,n in enumerate(CLASS_NAMES)}
ID_TO_LABEL={i:n for n,i in LABEL_TO_ID.items()}

if RUN_MODE=="GATE_CHECK":
    MAIN_EPOCHS=4
    RUN_MULTI_SEED=False
    RUN_ABLATION=False
    RUN_ROBUSTNESS=False
else:
    MAIN_EPOCHS=6
    RUN_MULTI_SEED=True
    RUN_ABLATION=True
    RUN_ROBUSTNESS=True

RESULT_DIR=Path("urlfusionnet_adaptive_final")
RESULT_DIR.mkdir(exist_ok=True)

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_all_seeds(42)
print("Mode:",RUN_MODE,"GPU:",gpus,"Batch:",BATCH_SIZE)


## 2. Kaggle dataset, audit, and fixed split

In [ ]:

KAGGLE_DATASET="sid321axn/malicious-urls-dataset"
EXPECTED_COUNTS={
    "benign":428103,
    "defacement":96457,
    "phishing":94111,
    "malware":32520,
}

candidates=[
    "/kaggle/input/malicious-urls-dataset/malicious_phish.csv",
    "./malicious_phish.csv",
]
csv_path=next((p for p in candidates if os.path.exists(p)),None)

if csv_path is None:
    import kagglehub
    root=kagglehub.dataset_download(KAGGLE_DATASET)
    matches=glob.glob(os.path.join(root,"**","malicious_phish.csv"),recursive=True)
    if not matches:
        matches=glob.glob(os.path.join(root,"**","*.csv"),recursive=True)
    if not matches: raise FileNotFoundError("Dataset CSV not found.")
    csv_path=matches[0]

raw=pd.read_csv(csv_path)
cols={c.lower().strip():c for c in raw.columns}
url_col=next((cols[k] for k in ["url","urls","link"] if k in cols),None)
label_col=next((cols[k] for k in ["type","label","class","category"] if k in cols),None)

df=raw[[url_col,label_col]].copy()
df.columns=["url","label"]
df["url"]=df["url"].astype("string")
df["label"]=df["label"].astype("string").str.strip().str.lower()

observed={k:int(df["label"].value_counts().get(k,0)) for k in EXPECTED_COUNTS}
assert observed==EXPECTED_COUNTS, observed

df=df.dropna(subset=["url","label"]).copy()
df["url"]=df["url"].str.strip()
df=df[df["url"].str.len()>0]
df=df[df["label"].isin(CLASS_NAMES)].copy()

card=df.groupby("url",sort=False)["label"].nunique()
conflict_urls=card[card>1].index
df=df[~df["url"].isin(conflict_urls)].copy()
df=df.drop_duplicates(subset=["url","label"],keep="first").copy()
df["label_id"]=df["label"].map(LABEL_TO_ID).astype("int32")

train_df,temp_df=train_test_split(
    df,test_size=0.20,random_state=SPLIT_SEED,stratify=df["label_id"]
)
val_df,test_df=train_test_split(
    temp_df,test_size=0.50,random_state=SPLIT_SEED,stratify=temp_df["label_id"]
)
for x in [train_df,val_df,test_df]: x.reset_index(drop=True,inplace=True)

assert len(set(train_df.url)&set(val_df.url))==0
assert len(set(train_df.url)&set(test_df.url))==0
assert len(set(val_df.url)&set(test_df.url))==0

counts=train_df["label_id"].value_counts().sort_index()
CLASS_WEIGHT={
    k:float(len(train_df)/(NUM_CLASSES*counts.loc[k]))
    for k in range(NUM_CLASSES)
}

print("Clean:",len(df))
print("Train/Val/Test:",len(train_df),len(val_df),len(test_df))
print("Class weights:",CLASS_WEIGHT)


## 3. Training-only vectorizers

In [ ]:

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
def url_token_standardize(x):
    x=tf.strings.lower(x)
    x=tf.strings.regex_replace(x,r"[^a-z0-9]+"," ")
    return tf.strings.strip(x)

char_vectorizer=layers.TextVectorization(
    max_tokens=MAX_CHAR_VOCAB,standardize=None,split="character",
    output_mode="int",output_sequence_length=MAX_CHAR_LEN
)
token_vectorizer=layers.TextVectorization(
    max_tokens=MAX_TOKEN_VOCAB,standardize=url_token_standardize,
    split="whitespace",output_mode="int",
    output_sequence_length=MAX_TOKEN_LEN
)

adapt_ds=tf.data.Dataset.from_tensor_slices(
    train_df["url"].astype(str).values
).batch(4096)
char_vectorizer.adapt(adapt_ds)
token_vectorizer.adapt(adapt_ds)

AUTOTUNE=tf.data.AUTOTUNE

def make_dataset(frame,training=False,seed=42):
    ds=tf.data.Dataset.from_tensor_slices((
        frame["url"].astype(str).values,
        frame["label_id"].astype("int32").values
    ))
    if training:
        ds=ds.shuffle(min(len(frame),200000),seed=seed,reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def make_predict_dataset(urls):
    return tf.data.Dataset.from_tensor_slices(
        np.asarray(urls,dtype=str)
    ).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds=make_dataset(val_df,False)


## 4. Stable layers and metric

In [ ]:

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class PositionalEmbedding(layers.Layer):
    def __init__(self,max_length,dim,**kwargs):
        super().__init__(**kwargs); self.max_length=max_length; self.dim=dim
    def build(self,input_shape):
        self.positional=self.add_weight(
            name="positional_embedding",
            shape=(1,self.max_length,self.dim),
            initializer="random_normal",trainable=True
        )
    def call(self,x):
        return x+self.positional[:,:tf.shape(x)[1],:]

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class AbsDifference(layers.Layer):
    def call(self,inputs):
        a,b=inputs; return tf.abs(a-b)

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class AdaptiveWeightedFusion(layers.Layer):
    def call(self,inputs):
        zc,zt,a=inputs
        return a[:,0:1]*zc+a[:,1:2]*zt

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class SafeMaskedMean(layers.Layer):
    def call(self,inputs):
        x,m=inputs
        m=tf.cast(m,x.dtype)[...,None]
        return tf.reduce_sum(x*m,axis=1)/tf.maximum(
            tf.reduce_sum(m,axis=1),tf.cast(1.0,x.dtype)
        )

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class SparseMacroF1(tf.keras.metrics.Metric):
    def __init__(self,num_classes=4,name="macro_f1",**kwargs):
        super().__init__(name=name,**kwargs)
        self.num_classes=num_classes
        self.cm=self.add_weight(
            name="cm",shape=(num_classes,num_classes),
            initializer="zeros",dtype=tf.float32
        )
    def update_state(self,y_true,y_pred,sample_weight=None):
        yt=tf.cast(tf.reshape(y_true,[-1]),tf.int32)
        yp=tf.argmax(y_pred,axis=-1,output_type=tf.int32)
        self.cm.assign_add(tf.math.confusion_matrix(
            yt,yp,num_classes=self.num_classes,dtype=tf.float32
        ))
    def result(self):
        tp=tf.linalg.diag_part(self.cm)
        fp=tf.reduce_sum(self.cm,axis=0)-tp
        fn=tf.reduce_sum(self.cm,axis=1)-tp
        p=tf.math.divide_no_nan(tp,tp+fp)
        r=tf.math.divide_no_nan(tp,tp+fn)
        f=tf.math.divide_no_nan(2*p*r,p+r)
        return tf.reduce_mean(f)
    def reset_state(self):
        self.cm.assign(tf.zeros_like(self.cm))



## 5. Final architecture

For the proposed model the adaptive gate is conditioned jointly on the projected character and token vectors:

\[
\mathbf{s}_i=[\mathbf{z}_i^c\parallel\mathbf{z}_i^t],
\]

\[
\boldsymbol{\alpha}_i=
\operatorname{softmax}
\left(
\mathbf W_g\,\phi(\mathbf W_s\mathbf s_i+\mathbf b_s)+\mathbf b_g
\right),
\]

\[
\mathbf f_i=
\alpha_i^c\mathbf z_i^c+
\alpha_i^t\mathbf z_i^t.
\]

The earlier element-wise product and absolute-difference terms are evaluated only as an explicit-interaction ablation.


In [ ]:

def build_model(variant="proposed"):
    url=keras.Input(shape=(),dtype=tf.string,name="url")

    # character branch
    cids=char_vectorizer(url)
    xc=layers.Embedding(
        len(char_vectorizer.get_vocabulary()),CHAR_EMB_DIM,name="char_embedding"
    )(cids)
    pools=[]
    for k in (3,5,7):
        h=layers.Conv1D(CONV_FILTERS,k,padding="same",use_bias=False,
                        name=f"char_conv_k{k}")(xc)
        h=layers.BatchNormalization(name=f"char_bn_k{k}")(h)
        h=layers.Activation(tf.nn.gelu,name=f"char_gelu_k{k}")(h)
        pools.append(layers.GlobalMaxPooling1D(name=f"char_gmp_k{k}")(h))
    hc=layers.Concatenate(name="char_multiscale_concat")(pools)
    hc=layers.Dense(LATENT_DIM,name="char_scale_projection")(hc)

    # token branch
    tids=token_vectorizer(url)
    token_mask=layers.Lambda(
        lambda ids:tf.not_equal(ids,0),name="token_valid_mask"
    )(tids)
    xt=layers.Embedding(
        len(token_vectorizer.get_vocabulary()),TOKEN_EMB_DIM,
        mask_zero=False,name="token_embedding"
    )(tids)
    xt=PositionalEmbedding(MAX_TOKEN_LEN,TOKEN_EMB_DIM,name="token_position")(xt)
    key_mask=layers.Lambda(
        lambda m:tf.expand_dims(m,axis=1),name="token_key_mask"
    )(token_mask)
    attn=layers.MultiHeadAttention(
        num_heads=ATTN_HEADS,key_dim=TOKEN_EMB_DIM//ATTN_HEADS,
        dropout=0.10,name="token_self_attention"
    )(xt,xt,attention_mask=key_mask)
    xt=layers.LayerNormalization(name="token_attn_ln")(layers.Add()([xt,attn]))
    ff=layers.Dense(TOKEN_EMB_DIM*2,activation=tf.nn.gelu,name="token_ff1")(xt)
    ff=layers.Dropout(DROPOUT)(ff)
    ff=layers.Dense(TOKEN_EMB_DIM,name="token_ff2")(ff)
    xt=layers.LayerNormalization(name="token_ff_ln")(layers.Add()([xt,ff]))
    ht=SafeMaskedMean(name="token_safe_pool")([xt,token_mask])

    # common space
    zc=layers.LayerNormalization(name="z_char")(
        layers.Dense(LATENT_DIM,name="char_common_projection")(hc)
    )
    zt=layers.LayerNormalization(name="z_token")(
        layers.Dense(LATENT_DIM,name="token_common_projection")(ht)
    )

    if variant=="char_only":
        fused=zc
    elif variant=="token_only":
        fused=zt
    elif variant=="fixed_avg":
        fused=layers.Average(name="fixed_average_fusion")([zc,zt])
    else:
        if variant=="explicit_interaction":
            prod=layers.Multiply(name="cross_product")([zc,zt])
            diff=AbsDifference(name="cross_absdiff")([zc,zt])
            gate_input=layers.Concatenate(name="interaction_representation")(
                [zc,zt,prod,diff]
            )
        else:
            gate_input=layers.Concatenate(name="adaptive_gate_context")([zc,zt])

        gh=layers.Dense(
            LATENT_DIM//2,activation=tf.nn.gelu,name="gate_hidden"
        )(gate_input)
        alpha=layers.Dense(
            2,activation="softmax",dtype="float32",name="fusion_weights"
        )(gh)
        fused=AdaptiveWeightedFusion(name="adaptive_weighted_fusion")(
            [zc,zt,alpha]
        )

    use_residual = variant not in (
        "char_only","token_only","proposed_no_residual"
    )
    if use_residual:
        r=layers.Dense(LATENT_DIM*2,activation=tf.nn.gelu,name="residual_fc1")(fused)
        r=layers.Dropout(DROPOUT)(r)
        r=layers.Dense(LATENT_DIM,name="residual_fc2")(r)
        fused=layers.LayerNormalization(name="residual_ln")(layers.Add()([fused,r]))

    out=layers.Dense(
        NUM_CLASSES,activation="softmax",dtype="float32",name="classifier"
    )(fused)
    return keras.Model(url,out,name=f"URLFusionNet_{variant}")

def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=LEARNING_RATE,weight_decay=1e-4,clipnorm=1.0
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            SparseMacroF1(NUM_CLASSES,name="macro_f1")
        ]
    )
    return model


## 6. Training and evaluation helpers

In [ ]:

def callbacks(tag):
    return [
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.EarlyStopping(
            monitor="val_macro_f1",mode="max",
            patience=2,restore_best_weights=True,verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_macro_f1",mode="max",
            factor=0.5,patience=1,min_lr=1e-5,verbose=1
        ),
        keras.callbacks.CSVLogger(RESULT_DIR/f"history_{tag}.csv")
    ]

def train_variant(variant,seed,epochs):
    tf.keras.backend.clear_session()
    set_all_seeds(seed)
    model=compile_model(build_model(variant))
    model.fit(
        make_dataset(train_df,True,seed),
        validation_data=val_ds,
        epochs=epochs,
        class_weight=CLASS_WEIGHT,
        callbacks=callbacks(f"{variant}_seed{seed}"),
        verbose=1
    )
    return model

def metric_row(y,pred,name,seed):
    p,r,f,_=precision_recall_fscore_support(
        y,pred,average="macro",zero_division=0
    )
    return {
        "Model":name,"Seed":seed,
        "Accuracy":accuracy_score(y,pred),
        "Macro_Precision":p,
        "Macro_Recall":r,
        "Macro_F1":f,
        "Weighted_F1":f1_score(y,pred,average="weighted",zero_division=0),
        "MCC":matthews_corrcoef(y,pred)
    }

def evaluate(model,frame,name,seed,report=False):
    y=frame["label_id"].to_numpy()
    probs=model.predict(
        make_predict_dataset(frame["url"].astype(str).values),verbose=0
    )
    assert np.isfinite(probs).all()
    pred=probs.argmax(1)
    row=metric_row(y,pred,name,seed)
    if report:
        rep=classification_report(
            y,pred,target_names=CLASS_NAMES,output_dict=True,
            digits=6,zero_division=0
        )
        pd.DataFrame(rep).T.to_csv(
            RESULT_DIR/f"classification_report_{name}_seed{seed}.csv"
        )
    return row,probs,pred


## 7. GATE_CHECK — proposed adaptive fusion, seed 42

In [ ]:

proposed42=train_variant("proposed",42,MAIN_EPOCHS)
row42,prob42,pred42=evaluate(
    proposed42,test_df,"URLFusionNet-Adaptive",42,report=True
)
display(pd.DataFrame([row42]))

gate_model=keras.Model(
    proposed42.input,proposed42.get_layer("fusion_weights").output
)
alpha=gate_model.predict(
    make_predict_dataset(test_df["url"].astype(str).values),verbose=0
)
assert np.isfinite(alpha).all()

gate_df=pd.DataFrame({
    "label":test_df["label"].values,
    "alpha_character":alpha[:,0],
    "alpha_token":alpha[:,1],
})

eps=1e-8
entropy=-(alpha*np.log(alpha+eps)).sum(axis=1)/np.log(2.0)
gate_df["normalized_entropy"]=entropy

summary=gate_df.groupby("label")[
    ["alpha_character","alpha_token","normalized_entropy"]
].agg(["mean","std","min","max"])

display(summary)

quantiles=gate_df[
    ["alpha_character","alpha_token","normalized_entropy"]
].quantile([0.01,0.05,0.25,0.50,0.75,0.95,0.99])
display(quantiles)

print("Overall std(alpha_character):",gate_df.alpha_character.std())
print("Overall std(alpha_token)    :",gate_df.alpha_token.std())
print("Mean normalized gate entropy:",gate_df.normalized_entropy.mean())
print("Character-dominant alpha>0.90:",
      float((gate_df.alpha_character>0.90).mean()))
print("Token-dominant alpha_token>0.90:",
      float((gate_df.alpha_token>0.90).mean()))

gate_df.to_csv(RESULT_DIR/"adaptive_gate_weights.csv",index=False)
summary.to_csv(RESULT_DIR/"adaptive_gate_summary_by_class.csv")
quantiles.to_csv(RESULT_DIR/"adaptive_gate_quantiles.csv")

# Flag only extreme numerical collapse.
collapsed = (
    gate_df.alpha_character.std() < 1e-4
    or ((gate_df.alpha_character > 0.999).mean() > 0.99)
    or ((gate_df.alpha_token > 0.999).mean() > 0.99)
)
print("GATE COLLAPSED:",collapsed)

if RUN_MODE=="GATE_CHECK":
    assert not collapsed, (
        "Adaptive gate still collapsed. Stop here and send this output."
    )
    print("\nGATE CHECK PASSED. Change RUN_MODE='FINAL' and rerun all cells.")


## 8. FINAL multi-seed evaluation

In [ ]:

multi_rows=[row42]

if RUN_MULTI_SEED:
    for seed in [123,2025]:
        m=train_variant("proposed",seed,MAIN_EPOCHS)
        row,_,_=evaluate(
            m,test_df,"URLFusionNet-Adaptive",seed,report=True
        )
        multi_rows.append(row)
        m.save_weights(
            RESULT_DIR/f"URLFusionNet_Adaptive_seed{seed}.weights.h5"
        )
        del m
        gc.collect()

multi_df=pd.DataFrame(multi_rows)
display(multi_df)
multi_df.to_csv(RESULT_DIR/"multiseed_results.csv",index=False)

if RUN_MULTI_SEED:
    mean_std=multi_df.drop(columns=["Model","Seed"]).agg(["mean","std"]).T
    display(mean_std)
    mean_std.to_csv(RESULT_DIR/"multiseed_mean_std.csv")


## 9. FINAL ablation aligned to the selected architecture

In [ ]:

if RUN_ABLATION:
    rows=[]
    variants=[
        ("char_only","Character-only"),
        ("token_only","Token-only"),
        ("fixed_avg","Fixed-average fusion"),
        ("proposed_no_residual","Adaptive gate without residual"),
        ("explicit_interaction","Explicit product/difference gate"),
    ]

    for variant,name in variants:
        m=train_variant(variant,42,4)
        row,_,_=evaluate(m,test_df,name,42)
        rows.append(row)
        del m
        gc.collect()

    rows.append({**row42,"Model":"URLFusionNet-Adaptive (Proposed)"})
    abl=pd.DataFrame(rows).sort_values("Macro_F1",ascending=False)
    display(abl)
    abl.to_csv(RESULT_DIR/"final_ablation.csv",index=False)

    fig,ax=plt.subplots(figsize=(10,5))
    ax.bar(abl["Model"],abl["Macro_F1"])
    ax.set_ylabel("Macro-F1")
    ax.set_title("Final URLFusionNet Ablation")
    ax.tick_params(axis="x",rotation=35)
    fig.tight_layout()
    fig.savefig(RESULT_DIR/"fig_final_ablation.png",dpi=300,bbox_inches="tight")
    plt.show()


## 10. FINAL robustness

In [ ]:

LOOKALIKE={"a":"4","e":"3","i":"1","o":"0","s":"5",
           "A":"4","E":"3","I":"1","O":"0","S":"5"}

def _rng(text,salt):
    h=hashlib.md5((salt+"||"+str(text)).encode("utf-8","ignore")).hexdigest()
    return random.Random(int(h[:8],16))

def p_insert(u):
    u=str(u)
    if not u:return u
    r=_rng(u,"ins"); pos=r.randrange(1,max(2,len(u)))
    return u[:pos]+r.choice("abcdefghijklmnopqrstuvwxyz0123456789")+u[pos:]

def p_delete(u):
    u=str(u); cand=[i for i,c in enumerate(u) if c.isalnum()]
    if not cand:return u
    pos=_rng(u,"del").choice(cand)
    return u[:pos]+u[pos+1:]

def p_sub(u):
    u=str(u); cand=[i for i,c in enumerate(u) if c in LOOKALIKE]
    if not cand:return p_insert(u)
    pos=_rng(u,"sub").choice(cand)
    return u[:pos]+LOOKALIKE[u[pos]]+u[pos+1:]

def p_path(u):
    u=str(u)
    if "?" in u:
        b,q=u.split("?",1)
        return b.rstrip("/")+"/verify-update?"+q
    return u.rstrip("/")+"/verify-update"

def p_query(u):
    u=str(u)
    return u+("&" if "?" in u else "?")+"ref=secure_update"

if RUN_ROBUSTNESS:
    perturb={
        "Clean":lambda x:x,
        "Insertion":p_insert,
        "Deletion":p_delete,
        "Substitution":p_sub,
        "PathModification":p_path,
        "QueryManipulation":p_query,
    }
    y=test_df["label_id"].to_numpy()
    rows=[]
    for name,fn in perturb.items():
        urls=test_df["url"].map(fn).values
        probs=proposed42.predict(make_predict_dataset(urls),verbose=0)
        pred=probs.argmax(1)
        rows.append(metric_row(y,pred,name,42))
    rob=pd.DataFrame(rows)
    display(rob)
    rob.to_csv(RESULT_DIR/"robustness_results.csv",index=False)


## 11. Efficiency

In [ ]:

def efficiency(model,urls):
    urls=np.asarray(urls,dtype=str)
    ds=make_predict_dataset(urls)
    for _ in range(2):
        model.predict(ds.take(2),verbose=0)
    t0=time.perf_counter()
    model.predict(ds,verbose=0)
    elapsed=time.perf_counter()-t0
    size=sum(np.asarray(w).nbytes for w in model.get_weights())/(1024**2)
    return {
        "Parameters":model.count_params(),
        "Model_Size_MB":size,
        "Inference_ms_per_URL":1000*elapsed/len(urls),
        "Throughput_URL_per_s":len(urls)/elapsed,
        "Latency_Sample_N":len(urls)
    }

eff=pd.DataFrame([{
    "Model":"URLFusionNet-Adaptive",
    **efficiency(
        proposed42,
        test_df["url"].iloc[:min(20000,len(test_df))].values
    )
}])
display(eff)
eff.to_csv(RESULT_DIR/"efficiency_results.csv",index=False)


## 12. Baseline comparison from Experiment 2

In [ ]:

baseline=pd.DataFrame([
["LinearSVM",0.6918829548,0.4335853998,0.2496499497],
["RandomForest",0.8781195408,0.8488062342,0.7768411736],
["XGBoost",0.8510419266,0.8171847418,0.7400696590],
["MultiScaleCharCNN",0.9646088096,0.9506399753,0.9324809992],
["CharCNN",0.9713470177,0.9583641694,0.9440658526],
["TokenBiLSTM",0.9322903669,0.9289451013,0.8797808935],
["CNNTransformer_TCURLStyle",0.9722984777,0.9590648302,0.9459760655],
],columns=["Model","Accuracy","Macro_F1","MCC"])

proposed=pd.DataFrame([[
    "URLFusionNet-Adaptive",row42["Accuracy"],row42["Macro_F1"],row42["MCC"]
]],columns=baseline.columns)

comparison=pd.concat([baseline,proposed],ignore_index=True).sort_values(
    "Macro_F1",ascending=False
)
display(comparison)
comparison.to_csv(RESULT_DIR/"baseline_comparison.csv",index=False)

best=baseline.loc[baseline.Macro_F1.idxmax()]
print("Best baseline:",best.Model)
print("Macro-F1 delta (percentage points):",
      100*(row42["Macro_F1"]-best.Macro_F1))


## 13. Final readiness

In [ ]:

checks={
    "Proposed seed42 finite":np.isfinite(row42["Macro_F1"]),
    "Gate not collapsed":not collapsed,
    "Gate summary saved":(RESULT_DIR/"adaptive_gate_summary_by_class.csv").exists(),
    "Efficiency saved":(RESULT_DIR/"efficiency_results.csv").exists(),
}

if RUN_MODE=="FINAL":
    checks.update({
        "Three seeds completed":len(multi_df)==3,
        "Ablation saved":(RESULT_DIR/"final_ablation.csv").exists(),
        "Robustness saved":(RESULT_DIR/"robustness_results.csv").exists(),
    })

ready=pd.DataFrame(
    [{"Requirement":k,"Passed":bool(v)} for k,v in checks.items()]
)
display(ready)
ready.to_csv(RESULT_DIR/"readiness.csv",index=False)

print("\nOutputs:")
for p in sorted(RESULT_DIR.glob("*")):
    print(" -",p)



## Paper interpretation

If `GATE_CHECK` passes and the FINAL run confirms the result across seeds:

- the final proposed mechanism should be described as **sample-adaptive cross-granularity character–token fusion**;
- the explicit Hadamard-product and absolute-difference interaction terms should **not** remain part of the final proposed method;
- they may be reported as a diagnostic architectural alternative if useful;
- the manuscript equations, Table III, Fig. 1, and Fig. 2 must be updated consistently after the final numbers are locked.
